# Auditando as partidas do fornecedor

## Demanda da aula 2

Com o ambiente no ar e os arquivos do fornecedor lidos, chegou o primeiro pedido do time de produto: estatísticas de
cada Copa e de cada seleção para usar nas transmissões. Durante um jogo, o narrador quer poder dizer coisas como
"esta é a Copa com a maior média de gols" ou "esta seleção tem o melhor saldo de gols em Copas".

Antes de calcular qualquer número, vem a pergunta que todo engenheiro de dados faz: **posso confiar nesse dado?** Na
aula 1, o público já chegou como texto, por causa da vírgula de milhar. Será que é o único problema? É hora de
conferir o arquivo com calma: qualquer erro no dado vai parar na transmissão.

Nesta aula, você audita e limpa as partidas, corrige os tipos, calcula as estatísticas e confere os outros
dois arquivos do fornecedor, as escalações e as Copas, contra as partidas.

| Etapa | O que fazer | Notebook |
|---|---|---|
| Task 1 | Auditar as partidas: quantas chegaram, quantas estão vazias ou repetidas e quantas sobram | `01_transformacao_vs_acao` |
| Task 2 | Padronizar os nomes das colunas e corrigir os tipos do público e da data | `02_select_filter_withcolumn` |
| Task 3 | Calcular as estatísticas por Copa e por seleção | `03_agregacoes` |
| Task 4 | Conferir as escalações e o resumo oficial de cada Copa e levar o código para o projeto | `04_join` |

**Pronto quando:** as estatísticas por Copa e por seleção chegam ao mesmo total de gols, nenhuma conversão de tipo
gerou nulos sem a gente saber, e a limpeza das partidas está num arquivo do projeto, `src/wc_pipeline/transformacoes.py`.

## Task 1: auditar as partidas

A conferência de volume que se faz em todo arquivo novo:

1. **Total bruto:** quantas linhas chegaram.
2. **Linhas vazias:** quantas linhas não têm o identificador da partida (`MatchID`).
3. **Duplicatas:** quantas linhas repetem uma partida (o mesmo `MatchID`) que já apareceu.
4. **Total limpo:** quantas partidas sobram depois de tirar as vazias e as repetidas.

A partir desta aula, a célula da sessão já vem pronta: o código do vídeo 1.4, com outro `appName`.

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-auditoria-partidas")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

## Lendo as partidas

In [2]:
from wc_pipeline.schemas import MATCHES

partidas_brutas = (
    spark.read
    .option("header", "true")
    .schema(MATCHES)
    .csv(f"{RAW}/WorldCupMatches.csv")
)

## Por que nada aconteceu?

A célula terminou na hora, e a aba **Jobs** da Spark UI não ganhou nenhum job: o Spark ainda não leu o arquivo, só
anotou o pedido num **plano**. Como o driver do vídeo 1.4, ele só manda trabalho para os executors quando você pede
um **resultado**.

| Tipo | O que faz | Como reconhecer | Exemplos |
|---|---|---|---|
| **Transformação** | acrescenta um passo ao plano | devolve **outro DataFrame** | `select`, `filter`, `distinct`, `dropDuplicates` |
| **Ação** | o Spark executa o plano | devolve **um resultado** (linhas na tela, um número) | `show`, `count` |

Esse jeito de trabalhar é a **lazy evaluation** (avaliação preguiçosa).

## A primeira ação: show

In [3]:
partidas_brutas\
    .select("Year", "MatchID", "Home Team Name", "Home Team Goals", "Away Team Goals", "Away Team Name")\
    .show()

+----+-------+--------------+---------------+---------------+--------------+
|Year|MatchID|Home Team Name|Home Team Goals|Away Team Goals|Away Team Name|
+----+-------+--------------+---------------+---------------+--------------+
|1930|      1|        France|              4|              1|        Mexico|
|1930|      2| United States|              3|              0|       Belgium|
|1930|      3|    Yugoslavia|              2|              1|        Brazil|
|1930|      4|       Romania|              3|              1|          Peru|
|1930|      5|     Argentina|              1|              0|        France|
|1930|      6|         Chile|              3|              0|        Mexico|
|1930|      7|    Yugoslavia|              4|              0|       Bolivia|
|1930|      8| United States|              3|              0|      Paraguay|
|1930|      9|       Uruguay|              1|              0|          Peru|
|1930|     10|         Chile|              1|              0|        France|

Um job novo: o `select` entrou no plano, e o `show` fez o plano rodar.

## 1. Total bruto

`count()` conta as linhas: é uma ação.

In [4]:
total_bruto = partidas_brutas.count()
total_bruto

977

## 2. Linhas vazias

O pedido: as linhas com `MatchID` nulo. O `filter` é uma transformação:

In [5]:
df_sem_id = partidas_brutas.filter(F.col("MatchID").isNull())
df_sem_id

DataFrame[Year: int, Datetime: string, Stage: string, Stadium: string, City: string, Home Team Name: string, Home Team Goals: int, Away Team Goals: int, Away Team Name: string, Win conditions: string, Attendance: string, Half-time Home Goals: int, Half-time Away Goals: int, Referee: string, Assistant 1: string, Assistant 2: string, RoundID: int, MatchID: int, Home Team Initials: string, Away Team Initials: string]

In [6]:
linhas_vazias = df_sem_id.count()
print(f"linhas vazias: {linhas_vazias}")

df_sem_id.select("Year", "MatchID", "Home Team Name", "Away Team Name").show()

linhas vazias: 3
+----+-------+--------------+--------------+
|Year|MatchID|Home Team Name|Away Team Name|
+----+-------+--------------+--------------+
|NULL|   NULL|          NULL|          NULL|
|NULL|   NULL|          NULL|          NULL|
|NULL|   NULL|          NULL|          NULL|
+----+-------+--------------+--------------+



## 3. Duplicatas

Se cada partida aparece uma vez só, o número de linhas e o de `MatchID` diferentes são iguais. `distinct()` tira as
repetições (transformação).

In [7]:
partidas_brutas.count()

977

In [8]:
df_com_id = partidas_brutas.filter(F.col("MatchID").isNotNull())
df_com_id.count()

974

In [9]:
df_com_id.select("MatchID").distinct().count()

964

In [10]:
df_com_id = partidas_brutas.filter(F.col("MatchID").isNotNull())

linhas_com_id = df_com_id.count()
ids_distintos = df_com_id.select("MatchID").distinct().count()

print(linhas_com_id)
print(ids_distintos)
print(linhas_com_id - ids_distintos)

974
964
10


In [11]:
df_com_id.distinct().count()

964

## 4. Total limpo

In [12]:
from wc_pipeline.schemas import MATCHES

partidas_brutas = (
    spark.read
    .option("header", "true")
    .schema(MATCHES)
    .csv(f"{RAW}/WorldCupMatches.csv")
)

partidas_limpas = (
    partidas_brutas
    .filter(F.col("MatchID").isNotNull())
    .dropDuplicates(["MatchID"])
)

print(partidas_limpas.count())

964


## Para consultar depois

- **Outras ações:** `first()` e `take(n)` trazem 1 ou `n` linhas para o Python. `collect()` traz **todas** as linhas
  para a memória do driver, e a documentação avisa: use só quando o resultado for pequeno, porque com milhões de
  linhas ele pode derrubar a aplicação. O `toPandas()` tem o mesmo cuidado.
- **Cada ação executa o plano de novo.** Dois `count()` seguidos leem o arquivo duas vezes. Guardar um resultado para
  reaproveitar é o `cache()`, da aula 4.
- **`printSchema()` não é ação.** O schema já é conhecido quando o plano é montado, então nenhuma linha é lida.
- **Olhe sempre o que o método devolve.** `df.count()` devolve um número: é ação. Já `df.groupBy("ano").count()`,
  que você vai ver no vídeo 2.3, devolve outro DataFrame, com a contagem de cada grupo: é transformação.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [13]:
spark.stop()